<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/RNNs_By_Hand_basic.ipynb)

# RNNs by Hand
--------------------------------
**Dr. Dave Wanik - University of Connecticut**
Being able to count the trainable parameters by hand and describing the output shape of each layer will help you ensure that you actually know how these algorithms work. It will crystallize why you need to prep your data as 3D tensors.

Here's a cheat sheat for counting parms in deep learning models:
* **Link:** https://towardsdatascience.com/counting-no-of-parameters-in-deep-learning-models-by-hand-8f1716241889

And here's the blog with animated RNN, LSTM and GRU
* **Link:** https://towardsdatascience.com/animated-rnn-lstm-and-gru-ef124d06cf45

In [1]:
from tensorflow.keras.layers import Input, Dense, SimpleRNN, LSTM, GRU, Conv2D
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding

# reproducibility: same seed every run (numbers on CPU match exactly; a GPU may drift a little)
import keras
keras.utils.set_random_seed(5509)


# Dense Neural Networks
(or feed-forward neural networks, FFNN)

* i, input size
* h, size of hidden layer
* o, output size
For one hidden layer,

```
num_params
= connections between layers + biases in every layer
= (i×h + h×o) + (h+o)
```

The example on the webpage assumes you have an input, a hidden layer and an output. For our examples with RNNs, we will assume o=0, and just use i and h. See below.

<!-- LATEX-CARD -->
### ✏️ The dense layer, written out

$$
\text{Dense params} \;=\; \underbrace{n_{in}\,n_{out}}_{\text{weights}} \;+\; \underbrace{n_{out}}_{\text{biases}}
$$

$n_{in}$ is whatever arrives from the layer before (after an RNN, that's its $h$ red dots); $n_{out}$ is the number you type in `Dense(n_out)`. Every example below ends with `Dense(1)`, so that head is always $h + 1$.


### One Simple RNN
One simpleRNN layer followed by a dense layer.

* `g`, no. of FFNNs in a unit (RNN has 1, GRU has 3, LSTM has 4)
* `h`, size of hidden units
* `i`, dimension/size of input

Since every FFNN (DNN) has `h(h+i) + h` parameters, we have
num_params = `g × [h(h+i) + h]`

Recall that the SimpleRNN only has one 'gate' or FFNN (you can see this in the cell!)

![alt text](https://miro.medium.com/max/1928/1*xn5kA92_J5KLaKcP7BMRLA.gif)



<!-- LATEX-CARD -->
### ✏️ One formula for every recurrent layer

$$
\text{params} \;=\; g\,\big[\;\underbrace{h\,(h+i)}_{\text{weights}} \;+\; \underbrace{h}_{\text{biases}}\;\big]
$$

| Symbol | Meaning | In the animation |
| :-- | :-- | :-- |
| $g$ | little networks inside one cell: **SimpleRNN 1 · GRU 3 · LSTM 4** | the boxes inside a cell |
| $h$ | hidden units, the number you type in `SimpleRNN(h)` | red dots |
| $i$ | features per time step (**not** the look-back) | green dots |

Why $h(h+i)$: each of the $h$ units looks at the $i$ inputs **and** the $h$ hidden values handed over from the previous time step.

**One exception to remember:** Keras' `GRU` defaults to `reset_after=True`, which adds a second bias vector to each gate:

$$
\text{GRU params (Keras default)} \;=\; 3\,\big[\,h\,(h+i) + 2h\,\big]
$$

Set `reset_after=False` and the GRU goes back to the one formula above.

**The look-back never appears.** Ten steps or ten million, the cell reuses the same weights at every step.


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 2 — The vanilla RNN, one time step at a time
- Three features meet two hidden units (the red dots): a 5-input dense net with tanh.
- The handoff: each time step's hidden state feeds the next, so the final state has seen the whole window.
- SimpleRNN(units) sets the red-dot count, and the output shape is ALWAYS units - independent of look-back.
- Data prep is the whole game: (samples, look-back, features) - the deck of cards.
-->


## One Simple RNN Layer (basic)

In [2]:
# here's the script for the image above

# for an SimpleRNN, the input shape is "input_shape=(n_steps, n_features)"
# this corresponds to the graph in "Animated!"
n_steps=50 # doesn't matter!
n_features=3 # the 3 green dots... APPL, GOOGLE, FB
model = Sequential()
# parms in SimpleRNN is
model.add((SimpleRNN(2, activation='relu', input_shape=(n_steps, n_features)))) # the two red dots

# it is those 2 red dots that will go into the dense layer (don't forget to add 1 for the bias!)
model.add(Dense(1)) # this dense layer is not show in the animation, but it's needed! # predict netflix!
model.summary()

C:\Users\dww05002\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 2)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15 (60.00 B)

 Trainable params: 15 (60.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ The math, written out
**Example 1 · SimpleRNN(2) on 3 features** · `input_shape=(50, 3)`

$$
\begin{aligned}
\textbf{SimpleRNN}(2) & = g\,[\,h(h+i) + h\,] = 1\,[\,2(2+3) + 2\,] = 1\,[\,10 + 2\,] = \mathbf{12} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 2\cdot 1 + 1 = \mathbf{3} \\
\textbf{Total} & = 12 + 3 = \mathbf{15}
\end{aligned}
$$

$g = 1$ (one little network), $h = 2$ (the red dots), $i = 3$ (the green dots: **features, not the look-back**).

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `SimpleRNN(2)` | `(None, 2)` | only the last hidden state |
| `Dense(1)` | `(None, 1)` | a plain dense layer |


In [3]:
# try the math
# for each layer


# TRAINABLE PARAMETERS
# the general RNN layer formula is g × [h(h+i) + h]
g = 1 # there's only 1 FFNN in a simpleRNN cell (look above!)
h = 2
i = 3 # this is number of features, not the lookback!
print(g*(h*(h+i) + h))

# SHAPE
# (None, 2) where 2 are the number of hidden units
# so the time series is now just a flattened input of 2 going into a dense layer
# this is what the 2 in simpleRNN(2) means! just 2 red dots.


# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 2 # 2 hidden node inputs
o = 0 # there is no output
print((i*h + h*o) + (h+o))

# output shape is (NONE,1)

12
3


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 3 — Trainable parameters and output shape of a SimpleRNN
- Cell = (features + units) x units + bias. H=2, I=3 -> 12; plus a 1-unit dense head (3) = 15.
- Bigger: 30 features, 25 units -> (30+25) x 25 + 25 = 1,400; plus the dense = 1,426.
- The general formula: G x [H(H+I) + H]. G = number of little networks in the cell; SimpleRNN G=1.
- Match every number to model.summary() on screen - Assignment 5 grades exactly this.
-->


## One Simple RNN Layer (advanced)

In [4]:
# here's a related quiz question

# for an SimpleRNN, the input shape is "input_shape=(n_steps, n_features)"
# this corresponds to the graph in "Animated!"
n_steps=50
n_features=30 # having 30 stocks for covariates
model = Sequential()
model.add((SimpleRNN(25, activation='relu', input_shape=(n_steps, n_features))))
# it is those 25 red dots going into the dense layer, so you need 26 parms
model.add(Dense(1))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 25)             │         1,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,426 (5.57 KB)

 Trainable params: 1,426 (5.57 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ The math, written out
**Example 2 · SimpleRNN(25) on 30 features** · `input_shape=(50, 30)`

$$
\begin{aligned}
\textbf{SimpleRNN}(25) & = g\,[\,h(h+i) + h\,] = 1\,[\,25(25+30) + 25\,] = 1\,[\,1{,}375 + 25\,] = \mathbf{1{,}400} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 25\cdot 1 + 1 = \mathbf{26} \\
\textbf{Total} & = 1{,}400 + 26 = \mathbf{1{,}426}
\end{aligned}
$$

Same formula, bigger numbers: $g = 1$, $h = 25$, $i = 30$. The look-back (50) never appears.

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `SimpleRNN(25)` | `(None, 25)` | only the last hidden state |
| `Dense(1)` | `(None, 1)` | a plain dense layer |


In [5]:
# try the math
# for each layer


# TRAINABLE PARAMETERS
# the general RNN layer formula is g × [h(h+i) + h]
g = 1
h = 25
i = 30 # this is number of features, not the lookback!
print(g*(h*(h+i) + h)) #answer = 1400

# SHAPE
# (None, 25) where 25 are the number of hidden units
# so the time series is now just a flattened input of 25 going into a dense layer
# this is what the 25 in simpleRNN(25) means! just 25 red dots.


# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 25
o = 0 # there is no output
print((i*h + h*o) + (h+o)) #answer = 26

# output shape is (NONE,1)

1400
26


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 4 — LSTM by hand: four networks and a cell state
- G=4: four networks, plus a CELL state (long memory) beside the HIDDEN state (recent memory) - that's what fixes the vanishing gradient.
- 5 inputs x 2 units + 2 bias = 12 per network, x4 = 48. Same formula, G=4.
- Why it's slow: every time step spins all four networks.
- Forget / input / output gates decide what to keep - but at heart it's four nets fit at once.
-->


## One LSTM Layer (basic)
Here is what an LSTM looks like. Recall that it has four 'gates' or FFNNs.

![alt text](https://miro.medium.com/max/2250/1*goJVQs-p9kgLODFNyhl9zA.gif)

In [6]:
# here is the code that corresponds to the image

# for an LSTM, the input shape is "input_shape=(n_steps, n_features)"
# same example as above, just presented a different way
n_steps= 50 # doesn't matter! it will loop.
n_features= 3 # these are the 3 green dots
model = Sequential()
model.add((LSTM(2,  # these are the 2 red dots
                activation='relu', input_shape=(n_steps, n_features))))
model.add(Dense(1)) # not shown, but you need it and should realize that the
                    # 2 dark red dots are what will go into the dense layer
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 2)              │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51 (204.00 B)

 Trainable params: 51 (204.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ The math, written out
**Example 3 · LSTM(2) on 3 features** · `input_shape=(50, 3)`

$$
\begin{aligned}
\textbf{LSTM}(2) & = g\,[\,h(h+i) + h\,] = 4\,[\,2(2+3) + 2\,] = 4\,[\,10 + 2\,] = \mathbf{48} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 2\cdot 1 + 1 = \mathbf{3} \\
\textbf{Total} & = 48 + 3 = \mathbf{51}
\end{aligned}
$$

Same picture as Example 1, but an LSTM cell holds **four** little networks, so $g = 4$: exactly $4\times$ the SimpleRNN's 12.

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `LSTM(2)` | `(None, 2)` | only the last hidden state |
| `Dense(1)` | `(None, 1)` | a plain dense layer |


In [7]:
# try the math

# simple RNN
# TRAINABLE PARAMETERS
# the generic RNN layer is g × [h(h+i) + h]
g = 4 #LSTM has 4 FFNNs!
h = 2 # hidden units within LSTM, the two red dots
i = 3 # this is number of features, not the lookback! these are your 3 stocks (green dots!)
print(g*(h*(h+i) + h))

# SHAPE
# (None, 2) where 2 are the number of hidden units
# so the time series is now just a flattened input of 2 going into a dense layer

# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has 1 output
i = 2 # these are all 4 inputs going into a dense layer
o = 0 # there is no output
print((i*h + h*o) + (h+o))

48
3


## One LSTM Layer (advanced)

In [8]:
# for an LSTM, the input shape is "input_shape=(n_steps, n_features)"
# same example as above, just presented a different way
n_steps= 30 # lookback
n_features= 5 # 5 different stocks, 5 green dots
model = Sequential()
model.add((LSTM(4, activation='relu', input_shape=(n_steps, n_features)))) # hidden units = 4 means 4 red dots
model.add(Dense(1))
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 4)              │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165 (660.00 B)

 Trainable params: 165 (660.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ The math, written out
**Example 4 · LSTM(4) on 5 features** · `input_shape=(30, 5)`

$$
\begin{aligned}
\textbf{LSTM}(4) & = g\,[\,h(h+i) + h\,] = 4\,[\,4(4+5) + 4\,] = 4\,[\,36 + 4\,] = \mathbf{160} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 4\cdot 1 + 1 = \mathbf{5} \\
\textbf{Total} & = 160 + 5 = \mathbf{165}
\end{aligned}
$$

$g = 4$, $h = 4$, $i = 5$.

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `LSTM(4)` | `(None, 4)` | only the last hidden state |
| `Dense(1)` | `(None, 1)` | a plain dense layer |


In [9]:
# try the math

# TRAINABLE PARAMETERS
# the generic RNN forumla is g × [h(h+i) + h]
g = 4 #LSTM has 4! these are the 4 FFNNs
h = 4 # hidden units within LSTM (you get to decide this! it's the red dots...)
i = 5 # this is number of features, not the lookback! the green dots... your 5 stocks
print(g*(h*(h+i) + h)) #answer = 160

# SHAPE
# (None, 4) where 4 are the number of hidden units
# so the time series is now just a flattened input of 4 going into a dense layer

# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 4 # these are all 4 inputs going into a dense layer
o = 0 # there is no output
print((i*h + h*o) + (h+o)) #answer = 5

160
5


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 5 — GRU by hand, and stacking/mixing cells
- G=3 (reset and update gates), counted the same way x3; reset_after=False makes Keras match the hand math - say it so summary() doesn't contradict you.
- Then the appendix: stack cells, mix SimpleRNN -> LSTM -> GRU - one-word swaps in Keras, everything else identical.
- Keep it under 8 minutes: the 2022 LSTM+GRU video ran 10:06 - this is its second half.
-->


## One GRU Layer (basic)
This is what a GRU looks like - note that it has three 'gates'.

![alt text](https://miro.medium.com/max/2214/1*lNNJOWnMjxLzdUnUQqwKcw.gif)

Caution: TensorFlow version difference!
Link: https://stackoverflow.com/questions/57318930/calculating-the-number-of-parameters-of-a-gru-layer-keras

Be careful of the bias term! Otherwise you need to add a second bias vector per gate. Keras' GRU defaults to `reset_after=True` (two bias vectors); these first two examples set `reset_after=False` so the one formula works. The stacked examples further down use the default, and their cards show the $+2h$.


In [10]:
# here is the example from the image
# and here is a related example
n_steps=50 # doesn't matter
n_features=3 # three stocks (FB, APPL, GOOG), three green dots
model = Sequential()
model.add((GRU(2, activation='relu', input_shape=(n_steps, n_features), # 2 red dots
               reset_after=False)))  # try this as False - helps math work out
model.add(Dense(1))
model.summary()

# if you don't say reset_after = False, you should add the bias terms
# which are bias_shape = (2, 3 * self.units)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 2)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39 (156.00 B)

 Trainable params: 39 (156.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ The math, written out
**Example 5 · GRU(2) on 3 features, `reset_after=False`** · `input_shape=(50, 3)`

$$
\begin{aligned}
\textbf{GRU}(2) & = g\,[\,h(h+i) + h\,] = 3\,[\,2(2+3) + 2\,] = 3\,[\,10 + 2\,] = \mathbf{36} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 2\cdot 1 + 1 = \mathbf{3} \\
\textbf{Total} & = 36 + 3 = \mathbf{39}
\end{aligned}
$$

A GRU holds **three** little networks, so $g = 3$. With `reset_after=False` each has one bias vector, so the same formula works: $3\times$ the SimpleRNN's 12.

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `GRU(2)` | `(None, 2)` | only the last hidden state |
| `Dense(1)` | `(None, 1)` | a plain dense layer |


In [11]:
# here is the math for that example
# try the math

# gru
# TRAINABLE PARAMETERS
# the general RNN layer is g × [h(h+i) + h]
g = 3 #GRU has 3 FFNNs (this is ALWAYS TRUE for GRU)
h = 2 # hidden units within GRU (RED DOTS)
i = 3 # this is number of features, not the lookback! (GREEN DOTS)
print('# of trainable parms in gru_1 = ', g*(h*(h+i) + h))

# SHAPE
# (None, 2) where 2 are the number of hidden units
# so the time series is now just a flattened input of 2 going into a dense layer

# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 2 # these are all 2 inputs going into a dense layer
o = 0 # there is no output
print((i*h + h*o) + (h+o))

# of trainable parms in gru_1 =  36
3


## One GRU Layer (advanced)

In [12]:
# and here is a related example
n_steps=30000000 # so many time steps!
n_features=5 # five stocks = five green dots = FB, GOOG, APPL, GE, AMD
model = Sequential()
model.add((GRU(4, activation='relu', input_shape=(n_steps, n_features), # 4 red dots
               reset_after=False)))  # try this as False for no extra bias
model.add(Dense(1))
model.summary()

# if you don't say reset_after = False, you should add the bias terms
# which are bias_shape = (2, 3 * self.units)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 4)              │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125 (500.00 B)

 Trainable params: 125 (500.00 B)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ The math, written out
**Example 6 · GRU(4) on 5 features, `reset_after=False`** · `input_shape=(30,000,000, 5)`

$$
\begin{aligned}
\textbf{GRU}(4) & = g\,[\,h(h+i) + h\,] = 3\,[\,4(4+5) + 4\,] = 3\,[\,36 + 4\,] = \mathbf{120} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 4\cdot 1 + 1 = \mathbf{5} \\
\textbf{Total} & = 120 + 5 = \mathbf{125}
\end{aligned}
$$

Thirty million time steps and the count is still tiny: **parameters depend on $h$ and $i$ only.**

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `GRU(4)` | `(None, 4)` | only the last hidden state |
| `Dense(1)` | `(None, 1)` | a plain dense layer |


In [13]:
# try the math

# gru
# TRAINABLE PARAMETERS
# the general RNN layer is g × [h(h+i) + h]
g = 3 #GRU has 3!
h = 4 # hidden units within GRU
i = 5 # this is number of features, not the lookback!
print('# of trainable parms in gru_1 = ', g*(h*(h+i) + h))
print(g*(h*(h+i) + h))

# SHAPE
# (None, 4) where 4 are the number of hidden units
# so the time series is now just a flattened input of 4 going into a dense layer

# dense layer
# TRAINABLE PARAMETERS
# the dense layer is (i×h + h×o) + (h+o)
# but we ignore h since there is not output
h = 1 # the dense layer has a 1
i = 4 # these are all 4 inputs going into a dense layer
o = 0 # there is no output
print((i*h + h*o) + (h+o))

# of trainable parms in gru_1 =  120
120
5


# Advanced (stacking, mixing and matching.)
We will cover this in future lectures - provided as FYI.


### Two GRU layers going into a SimpleRNN
This is the fun part! Since you are returning sequences - the output shape will be 3D... you are storing all outputs from the DNNs within each GRU layer!

This is where n_steps actually gets used in the output size. Don't forget to set `return_sequences=True` when stacking layers - except for the last one that goes into the Dense layer.

In [14]:
n_steps=30 # this matters for output shape when we return sequences!
n_features=5 # these are 5 stocks (FB, APPL, GE, NETFLIX, AMD)

model = Sequential()
model.add((GRU(4, return_sequences=True, activation='relu', input_shape=(n_steps, n_features))))
model.add((GRU(2, return_sequences=True, activation='relu')))
model.add((SimpleRNN(25, activation='relu')))
model.add(Dense(1))
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ (None, 30, 4)          │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 30, 2)          │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 25)             │           700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 906 (3.54 KB)

 Trainable params: 906 (3.54 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ The math, written out
**Example 7 · stacked GRU → GRU → SimpleRNN (Keras default GRU)** · `input_shape=(30, 5)`

$$
\begin{aligned}
\textbf{GRU}(4) & = g\,[\,h(h+i) + 2h\,] = 3\,[\,4(4+5) + 2\cdot4\,] = 3\,[\,36 + 8\,] = \mathbf{132} \\
\textbf{GRU}(2) & = g\,[\,h(h+i) + 2h\,] = 3\,[\,2(2+4) + 2\cdot2\,] = 3\,[\,12 + 4\,] = \mathbf{48} \\
\textbf{SimpleRNN}(25) & = g\,[\,h(h+i) + h\,] = 1\,[\,25(25+2) + 25\,] = 1\,[\,675 + 25\,] = \mathbf{700} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 25\cdot 1 + 1 = \mathbf{26} \\
\textbf{Total} & = 132 + 48 + 700 + 26 = \mathbf{906}
\end{aligned}
$$

Two new rules. **(1) Each layer's $i$ is the previous layer's $h$** (5 → 4 → 2 → 25). **(2)** These GRUs use Keras' default `reset_after=True`, which gives every gate a *second* bias vector, so the GRU term becomes $3\,[\,h(h+i) + 2h\,]$.

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `GRU(4)` | `(None, 30, 4)` | `return_sequences=True` keeps every time step |
| `GRU(2)` | `(None, 30, 2)` | `return_sequences=True` keeps every time step |
| `SimpleRNN(25)` | `(None, 25)` | only the last hidden state |
| `Dense(1)` | `(None, 1)` | a plain dense layer |


In [15]:
# try the math - one layer at a time, passing each layer's h down as the next layer's i

# gru_1: GRU(4) on 5 features. Keras' default reset_after=True adds a 2nd bias vector per gate -> +2h
g, h, i = 3, 4, 5
gru_1 = g * (h * (h + i) + 2 * h)
print("gru_1      :", gru_1)        # 132   output shape (None, 30, 4) - return_sequences keeps all 30 steps

# gru_2: GRU(2); its input is gru_1's 4 hidden units
g, h, i = 3, 2, 4
gru_2 = g * (h * (h + i) + 2 * h)
print("gru_2      :", gru_2)        # 48    output shape (None, 30, 2)

# simple_rnn: SimpleRNN(25); its input is gru_2's 2 hidden units
g, h, i = 1, 25, 2
simple_rnn = g * (h * (h + i) + h)
print("simple_rnn :", simple_rnn)   # 700   output shape (None, 25) - no return_sequences, last step only

# dense: Dense(1) on the 25 red dots
n_in, n_out = 25, 1
dense = n_in * n_out + n_out
print("dense      :", dense)        # 26    output shape (None, 1)

print("total      :", gru_1 + gru_2 + simple_rnn + dense)   # 906


gru_1      : 132
gru_2      : 48
simple_rnn : 700
dense      : 26
total      : 906


### One SimpleRNN going into an LSTM
Left to students as an exercise.

In [16]:
n_steps=15 # lookback
n_features=30 # 30 different stocks

model = Sequential()
model.add((SimpleRNN(20, return_sequences=True, activation='relu', input_shape=(n_steps, n_features))))
model.add((LSTM(4, activation='relu'))) # see how there is NO RETURN SEQUENCES!!!
model.add(Dense(1))                             # you just keep the last hidden state
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 15, 20)         │         1,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 4)              │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,425 (5.57 KB)

 Trainable params: 1,425 (5.57 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Answer — the math, written out
*Try it yourself before reading on.*
**Example 8 · SimpleRNN → LSTM** · `input_shape=(15, 30)`

$$
\begin{aligned}
\textbf{SimpleRNN}(20) & = g\,[\,h(h+i) + h\,] = 1\,[\,20(20+30) + 20\,] = 1\,[\,1{,}000 + 20\,] = \mathbf{1{,}020} \\
\textbf{LSTM}(4) & = g\,[\,h(h+i) + h\,] = 4\,[\,4(4+20) + 4\,] = 4\,[\,96 + 4\,] = \mathbf{400} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 4\cdot 1 + 1 = \mathbf{5} \\
\textbf{Total} & = 1{,}020 + 400 + 5 = \mathbf{1{,}425}
\end{aligned}
$$

The LSTM's $i$ is 20: the SimpleRNN's 20 red dots, handed over at every time step.

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `SimpleRNN(20)` | `(None, 15, 20)` | `return_sequences=True` keeps every time step |
| `LSTM(4)` | `(None, 4)` | only the last hidden state |
| `Dense(1)` | `(None, 1)` | a plain dense layer |


### Monster #1
Left as an exercise for students.

In [17]:
n_steps=30
n_features=30
model = Sequential()
model.add((SimpleRNN(30, return_sequences=True, activation='relu', input_shape=(n_steps, n_features))))
model.add((GRU(30, return_sequences=True,activation='relu')))
model.add((LSTM(30,activation='relu')))
model.add((Dense(30,activation='relu')))
model.add(Dense(1))
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_4 (SimpleRNN)        │ (None, 30, 30)         │         1,830 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_4 (GRU)                     │ (None, 30, 30)         │         5,580 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 30)             │         7,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 30)             │           930 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,691 (61.29 KB)

 Trainable params: 15,691 (61.29 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Answer — the math, written out
*Try it yourself before reading on.*
**Example 9 · Monster #1** · `input_shape=(30, 30)`

$$
\begin{aligned}
\textbf{SimpleRNN}(30) & = g\,[\,h(h+i) + h\,] = 1\,[\,30(30+30) + 30\,] = 1\,[\,1{,}800 + 30\,] = \mathbf{1{,}830} \\
\textbf{GRU}(30) & = g\,[\,h(h+i) + 2h\,] = 3\,[\,30(30+30) + 2\cdot30\,] = 3\,[\,1{,}800 + 60\,] = \mathbf{5{,}580} \\
\textbf{LSTM}(30) & = g\,[\,h(h+i) + h\,] = 4\,[\,30(30+30) + 30\,] = 4\,[\,1{,}800 + 30\,] = \mathbf{7{,}320} \\
\textbf{Dense}(30) & = n_{in}\,n_{out} + n_{out} = 30\cdot 30 + 30 = \mathbf{930} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 30\cdot 1 + 1 = \mathbf{31} \\
\textbf{Total} & = 1{,}830 + 5{,}580 + 7{,}320 + 930 + 31 = \mathbf{15{,}691}
\end{aligned}
$$

Work top to bottom and pass $h$ down as the next $i$. The GRU uses the default `reset_after=True` ($+2h$).

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `SimpleRNN(30)` | `(None, 30, 30)` | `return_sequences=True` keeps every time step |
| `GRU(30)` | `(None, 30, 30)` | `return_sequences=True` keeps every time step |
| `LSTM(30)` | `(None, 30)` | only the last hidden state |
| `Dense(30)` | `(None, 30)` | a plain dense layer |
| `Dense(1)` | `(None, 1)` | a plain dense layer |


### Monster #2
Left as an exercise for students.

In [18]:
n_steps=50
n_features=40
model = Sequential()
model.add((SimpleRNN(30, return_sequences=True, activation='relu', input_shape=(n_steps, n_features))))
model.add((GRU(20, return_sequences=True,activation='relu')))
model.add((GRU(25, return_sequences=True,activation='relu')))
model.add((GRU(22, return_sequences=True,activation='relu')))
model.add((GRU(21, return_sequences=True,activation='relu')))
model.add((SimpleRNN(10,activation='relu')))
model.add((Dense(50,activation='relu')))
model.add((Dense(50,activation='relu')))
model.add((Dense(50,activation='relu')))
model.add((Dense(50,activation='relu')))
model.add(Dense(1))
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_5 (SimpleRNN)        │ (None, 50, 30)         │         2,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ (None, 50, 20)         │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_6 (GRU)                     │ (None, 50, 25)         │         3,525 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 50, 22)         │         3,234 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_8 (GRU)                     │ (None, 50, 21)         │         2,835 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_6 (SimpleRNN)        │ (None, 10)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 50)             │           550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,415 (91.46 KB)

 Trainable params: 23,415 (91.46 KB)

 Non-trainable params: 0 (0.00 B)

<!-- LATEX-CARD -->
### ✏️ Answer — the math, written out
*Try it yourself before reading on.*
**Example 10 · Monster #2** · `input_shape=(50, 40)`

$$
\begin{aligned}
\textbf{SimpleRNN}(30) & = g\,[\,h(h+i) + h\,] = 1\,[\,30(30+40) + 30\,] = 1\,[\,2{,}100 + 30\,] = \mathbf{2{,}130} \\
\textbf{GRU}(20) & = g\,[\,h(h+i) + 2h\,] = 3\,[\,20(20+30) + 2\cdot20\,] = 3\,[\,1{,}000 + 40\,] = \mathbf{3{,}120} \\
\textbf{GRU}(25) & = g\,[\,h(h+i) + 2h\,] = 3\,[\,25(25+20) + 2\cdot25\,] = 3\,[\,1{,}125 + 50\,] = \mathbf{3{,}525} \\
\textbf{GRU}(22) & = g\,[\,h(h+i) + 2h\,] = 3\,[\,22(22+25) + 2\cdot22\,] = 3\,[\,1{,}034 + 44\,] = \mathbf{3{,}234} \\
\textbf{GRU}(21) & = g\,[\,h(h+i) + 2h\,] = 3\,[\,21(21+22) + 2\cdot21\,] = 3\,[\,903 + 42\,] = \mathbf{2{,}835} \\
\textbf{SimpleRNN}(10) & = g\,[\,h(h+i) + h\,] = 1\,[\,10(10+21) + 10\,] = 1\,[\,310 + 10\,] = \mathbf{320} \\
\textbf{Dense}(50) & = n_{in}\,n_{out} + n_{out} = 10\cdot 50 + 50 = \mathbf{550} \\
\textbf{Dense}(50) & = n_{in}\,n_{out} + n_{out} = 50\cdot 50 + 50 = \mathbf{2{,}550} \\
\textbf{Dense}(50) & = n_{in}\,n_{out} + n_{out} = 50\cdot 50 + 50 = \mathbf{2{,}550} \\
\textbf{Dense}(50) & = n_{in}\,n_{out} + n_{out} = 50\cdot 50 + 50 = \mathbf{2{,}550} \\
\textbf{Dense}(1) & = n_{in}\,n_{out} + n_{out} = 50\cdot 1 + 1 = \mathbf{51} \\
\textbf{Total} & = 2{,}130 + 3{,}120 + 3{,}525 + 3{,}234 + 2{,}835 + 320 + 550 + 2{,}550 + 2{,}550 + 2{,}550 + 51 = \mathbf{23{,}415}
\end{aligned}
$$

Eleven layers, one habit: **the previous layer's units become this layer's inputs.** All four GRUs use the default `reset_after=True` ($+2h$).

| Layer | Output shape | Why |
| :-- | :-- | :-- |
| `SimpleRNN(30)` | `(None, 50, 30)` | `return_sequences=True` keeps every time step |
| `GRU(20)` | `(None, 50, 20)` | `return_sequences=True` keeps every time step |
| `GRU(25)` | `(None, 50, 25)` | `return_sequences=True` keeps every time step |
| `GRU(22)` | `(None, 50, 22)` | `return_sequences=True` keeps every time step |
| `GRU(21)` | `(None, 50, 21)` | `return_sequences=True` keeps every time step |
| `SimpleRNN(10)` | `(None, 10)` | only the last hidden state |
| `Dense(50)` | `(None, 50)` | a plain dense layer |
| `Dense(50)` | `(None, 50)` | a plain dense layer |
| `Dense(50)` | `(None, 50)` | a plain dense layer |
| `Dense(50)` | `(None, 50)` | a plain dense layer |
| `Dense(1)` | `(None, 1)` | a plain dense layer |
